In [32]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc', decode_timedelta=True) #***
# parcel1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc', decode_timedelta=True) #***
# res='1km';t_res='5min'
# Np_str='1e6'

# dx = 1km; Np = 50M; Nz = 95
#Importing Model Data
dir2='/home/air673/koa_scratch/'
data1=xr.open_dataset(dir2+'cm1out_1km_1min_95nz.nc', decode_timedelta=True) #***
parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_95nz.nc', decode_timedelta=True) #***
res='1km'; t_res='1min_95nz'; Np_str='50e6'

# # dx = 250m; Np = 50M
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_250m_1min_50M.nc', decode_timedelta=True) #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_250m_1min_50M.nc', decode_timedelta=True) #***
# res='250m'; t_res='1min'; Np_str='50e6'

In [33]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [34]:
####################################
# SETUP

In [35]:
################################
#ENTRAINMENT DATA_TYPE SETUP #TESTING
PROCESSING=False #if using full entrainment calculations
PROCESSING=True #if using turbulence-filtered entrainment calculations
################################

In [53]:
################################
#JOB ARRAY SETUP
################################
#*#*
# how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100
if Np_str=='1e6':
    num_jobs=60 #1M parcels
    num_slurm_jobs=10
elif Np_str=='50e6':
    num_jobs=400 #50M parcels
    num_slurm_jobs=100
##############################

In [37]:
##############################################
#DATA LOADING FUNCTIONS

In [38]:
#INITIALIZE DATA FUNCTION
###############################################################
def InitiateArray(out_file,vars,t_chunk_size,z_chunk_size,t_size=None,z_size=None):
    # Define array dimensions (adjust based on your data)

    if t_size==None:
        t_size = len(data1['time'])  # Number of timesteps
    if z_size==None:
        z_size = len(data1['zh'])    # Number of vertical levels
    
    with h5py.File(out_file, 'w') as f: 
        # Check if the dataset 'theta_e' already exists
        for var_name in vars:
            if var_name not in f:
                # Create a dataset with the full size for all time steps (initially empty)
                f.create_dataset(var_name, 
                                 (t_size, z_size),  # Full size for all timesteps
                                 chunks=(t_chunk_size, z_chunk_size))  # Chunks for time axis to allow resizing

def InitiateArray_Job(data,out_file,vars,t_chunk_size,z_chunk_size,t_size=None,z_size=None):
    # Define array dimensions (adjust based on your data)

    if t_size==None:
        t_size = len(data['time'])  # Number of timesteps
    if z_size==None:
        z_size = len(data1['zh'])    # Number of vertical levels
    
    with h5py.File(out_file, 'w') as f: 
        # Check if the dataset 'theta_e' already exists
        for var_name in vars:
            if var_name not in f:
                # Create a dataset with the full size for all time steps (initially empty)
                f.create_dataset(var_name, 
                                 (t_size, z_size),  # Full size for all timesteps
                                 chunks=(t_chunk_size, z_chunk_size))  # Chunks for time axis to allow resizing

In [39]:
#THIS FUNCTION IS FOR RUNNING WITH SLURM JOB ARRAY
#(SPLITS UP JOB_ARRAY BELOW INTO EVEN MORE TASKS)
def StartSlurmJobArray(num_jobs,num_slurm_jobs, ISRUN):
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    if ISRUN==False:
        start_job=1;end_job=num_jobs+1
        return start_job,end_job
    total_elements=num_jobs #total num of variables

    job_range = total_elements // num_slurm_jobs  # Base size for each chunk
    remaining = total_elements % num_slurm_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_slurm_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_slurm_jobs - 1: 
            end_job = total_elements 
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_slurm_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()

    # if sbatch==True:
        
    start_job, end_job = get_job_range(job_id, num_slurm_jobs)
    index_adjust=start_job
    # print(f'start_job = {start_job}, end_job = {end_job}')
    if start_job==0: start_job=1
    if end_job==total_elements: end_job+=1
    return start_job,end_job

# job_id=1
# [start_slurm_job,end_slurm_job,slurm_index_adjust]=StartSlurmJobArray(num_jobs,num_slurm_jobs,ISRUN)
# parcel=parcel1.isel(xh=slice(start_job,end_job))

In [40]:
#JOB ARRAY SETUP
def StartJobArray(job_id,num_jobs):
    total_elements=len(data1['time']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()

    # if sbatch==True:
    #     job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    #     if job_id==0: job_id=1
        
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    # print(f'start_job = {start_job}, end_job = {end_job}')
    return start_job,end_job,index_adjust
# job_id=150
# [start_job,end_job,index_adjust]=StartJobArray(job_id,num_jobs)

In [41]:
# Reading Back Data Later
##############
def make_data_dict(in_file,var_names,read_type,start_job,end_job):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][start_job:end_job] for var_name in var_names}

            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][start_job:end_job].compute().data for k in var_names}

    return data_dict

# read_type='xarray'
read_type='h5py'

In [42]:
def GetData(data1,parcel1,start_job,end_job):
    #Indexing Array with JobArray
    data=data1.isel(time=slice(start_job,end_job))
    parcel=parcel1.isel(time=slice(start_job,end_job))
    return data,parcel
# [data,parcel]=GetData(data1,parcel1,start_job,end_job)

In [43]:
def GetSpatialData(data,start_job,end_job):
    import h5py
    dir2=dir+'Project_Algorithms/Lagrangian_Arrays/OUTPUT/'
    in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'
    
    var_names = ['Z']#, 'Y', 'X']
    data_dict = make_data_dict(in_file,var_names,read_type,start_job,end_job)
    Z, = (data_dict[k] for k in var_names) #,Y,X
    
    # #Making Time Matrix
    Nt=len(data['time'])
    T = np.broadcast_to(np.arange(Nt)[:, None], Z.shape)  # shape: (Nt, p) #A_c==>Z
    
    # check_memory(globals())
    return T,Z#,Y,X

In [44]:
def GetAData(PROCESSING,start_job,end_job):
    if PROCESSING==False:
        dir2=dir+'Project_Algorithms/Lagrangian_Arrays/OUTPUT/'
        in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'
        
        var_names = ['A_g', 'A_c']
        data_dict = make_data_dict(in_file,var_names,read_type,start_job,end_job)
        A_g, A_c = (data_dict[k] for k in var_names) #, W
    
    elif PROCESSING==True:
        dir2=dir+'Project_Algorithms/Entrainment/OUTPUT/'
        in_file=dir2+f'processed_binary_arrays_{res}_{t_res}_{Np_str}.h5'
        
        var_names = ['A_g_Processed', 'A_c_Processed']
        data_dict = make_data_dict(in_file,var_names,read_type,start_job,end_job)
        A_g,A_c = (data_dict[k] for k in var_names)
        # check_memory(globals())
    return A_g,A_c

In [45]:
def GetAPrior(PROCESSING,start_job): #FIX FOR JOB_ARRAY (SO FIRST TIMESTEP WILL NOT BE BLANK)
    if start_job!=0:
        A_g_Prior,A_c_Prior=GetAData(PROCESSING,start_job-1,start_job);
        return A_g_Prior,A_c_Prior
    else:
        return 0,0
def GetAPost(PROCESSING,end_job): #*#*
    if end_job!=len(data1['time']):
        [A_g_Post,A_c_Post]=GetAData(PROCESSING,end_job-1,end_job+1)
        return A_g_Post,A_c_Post
    else:
        return 0,0
def FixDetrainment(A1_Post,A2_Post): #*#*
    #when using slurm_job_array, last timestep is not calculated
    #since data is outside of access scope
    #THIS FUNCTION FIXES THAT
    D3 = np.zeros_like(A2_Post,dtype=np.int8)
    D3[1:, :] = A2_Post[1:, :]*1 + A1_Post[:-1, :]*1
    return D3[1:]

In [46]:
###########################################################################################################################################################################

In [47]:
####################################
#Functions

In [48]:
def SubtractA(A,A_Prior):
    D = np.zeros_like(A,dtype=np.int8)
    D[1:, :] = A[1:, :]*1 - A[:-1, :]*1
    if start_job!=0:
        D[0, :] = A[0, :]*1-A_Prior*1 #FIX FOR FIRST TIMESTEP
    return D

def AddAs(A1,A2,A1_Prior):
    S = np.zeros_like(A2,dtype=np.int8)
    S[1:, :] = A2[1:, :]*1 + A1[:-1, :]*1
    if start_job!=0:
        S[0, :] = A2[0, :]*1 + A1_Prior*1 #FIX FOR FIRST TIMESTEP
    return S

def S_2d(data, A1,A2, T, Z, A1_Prior,A1_Post,A2_Post, type1): #*#*
    # start_time = time.time()
    
    #calculating entrainment going from general <==> cloudy updrafts
    S=AddAs(A1,A2,A1_Prior) #General <==> Cloudy
    cond=(S==2) #must be either entrained OR detrained from/to environment) AND (NOT move between general and cloudy updrafts) 
    D = cond*1 #converting to 1s and 0s

    #MAKING DETRAINMENT VERSION (SAME BUT STORED AT PREVIOUS TIMESTEP)
    if type1=='d':
        D2=np.zeros_like(D); D2[:-1]= D[1:]; D=D2.copy() #NEED TO SHIFT TO PREVIOUS TIME TO STORE D CORRECTLY 
        if not isinstance(A2_Post, int): D3=FixDetrainment(A1_Post,A2_Post);D[-1]=D3.copy(); #*#*

    # Initialize time and vertical dimension arrays
    Nt = len(data['time']); Nz = len(data1['zh'])
    # Initialize result array
    result = np.zeros((Nt, Nz))

    # Use np.add.at to accumulate values in the result array
    np.add.at(result, (T, Z), D)

    # end_time = time.time()
    # print(f"Execution time: {(end_time - start_time)} seconds")
    return result

In [49]:
def MakeDictionary(**vars):
    return vars
def RunCalculation(data, A_g,A_c, T,Z, A_g_Prior,A_c_Prior,A_g_Post,A_c_Post):    
    # Set A based on PROCESSING state
    
    profile_array_c_to_g_E = S_2d(data, A_c,A_g, T, Z, A1_Prior=A_c_Prior, A1_Post=A_c_Post,A2_Post=A_g_Post, type1='e')
    profile_array_g_to_c_E = S_2d(data, A_g,A_c, T, Z, A1_Prior=A_g_Prior, A1_Post=A_g_Post,A2_Post=A_c_Post, type1='e')
    profile_array_c_to_g_D = S_2d(data, A_c,A_g, T, Z, A1_Prior=A_c_Prior, A1_Post=A_c_Post,A2_Post=A_g_Post, type1='d')
    profile_array_g_to_c_D = S_2d(data, A_g,A_c, T, Z, A1_Prior=A_g_Prior, A1_Post=A_g_Post,A2_Post=A_c_Post, type1='d')

    VARs=MakeDictionary(profile_array_c_to_g_E=profile_array_c_to_g_E, 
                        profile_array_g_to_c_E=profile_array_g_to_c_E,
                        profile_array_c_to_g_D=profile_array_c_to_g_D, 
                        profile_array_g_to_c_D=profile_array_g_to_c_D)
    return VARs

In [50]:
def SaveProfile(VARs,job_id,PROCESSING):
    dir2=dir+'Project_Algorithms/Entrainment/SBATCH/job_out_3/'
    
    #SAVING
    # print('SAVING')
    if PROCESSING==False:
        out_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_{res}_{t_res}_{Np_str}'
    elif PROCESSING==True:
        out_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}'
    out_file+=f'_{job_id}.h5'
    
    vars=VARs.keys()
    t_chunk_size=end_job-start_job
    z_chunk_size=2
    InitiateArray_Job(data,out_file,vars,t_chunk_size=t_chunk_size,z_chunk_size=z_chunk_size)
    
    with h5py.File(out_file, 'a') as f: 
        for var in VARs:
            f[f'{var}'][:]=VARs[var]
    # print('done')

In [51]:
##############################################
#RUNNING

In [85]:
[start_slurm_job,end_slurm_job]=StartSlurmJobArray(num_jobs=num_jobs,num_slurm_jobs=num_slurm_jobs,ISRUN=True) #if ISRUN is False, then will not run using slurm_job_array

print(f"Running on Slurm_Jobs for Slurm_Job_Ids: {(start_slurm_job,end_slurm_job-1)}")

job_id_list=np.arange(start_slurm_job,end_slurm_job)
for job_id in job_id_list:
    if job_id % 1 == 0: print(f'current job_id = {job_id}')
    [start_job,end_job,index_adjust]=StartJobArray(job_id,num_jobs)

    #SLICING DATA
    [data,parcel]=GetData(data1,parcel1,start_job,end_job)
    [A_g,A_c]=GetAData(PROCESSING,start_job,end_job)
    [A_g_Prior,A_c_Prior]=GetAPrior(PROCESSING,start_job) #FIX FOR FIRST TIMESTEP
    [A_g_Post,A_c_Post]=GetAPost(PROCESSING,end_job) #FIX FOR LAST TIMESTEP #*#*#
    [T,Z]=GetSpatialData(data,start_job,end_job)

    #RUNNING CALCULATION
    VARs=RunCalculation(data, A_g,A_c, T,Z, A_g_Prior,A_c_Prior,A_g_Post,A_c_Post)

    #SAVING
    SaveProfile(VARs,job_id,PROCESSING)
    # check_memory(globals())
    del VARs

Running on Slurm_Jobs for Slurm_Job_Ids: (1, 60)
current job_id = 1
current job_id = 2
current job_id = 3
current job_id = 4
current job_id = 5
current job_id = 6
current job_id = 7
current job_id = 8
current job_id = 9
current job_id = 10
current job_id = 11
current job_id = 12
current job_id = 13
current job_id = 14
current job_id = 15
current job_id = 16
current job_id = 17
current job_id = 18
current job_id = 19
current job_id = 20
current job_id = 21
current job_id = 22
current job_id = 23
current job_id = 24
current job_id = 25
current job_id = 26
current job_id = 27
current job_id = 28
current job_id = 29
current job_id = 30
current job_id = 31
current job_id = 32
current job_id = 33
current job_id = 34
current job_id = 35
current job_id = 36
current job_id = 37
current job_id = 38
current job_id = 39
current job_id = 40
current job_id = 41
current job_id = 42
current job_id = 43
current job_id = 44
current job_id = 45
current job_id = 46
current job_id = 47
current job_id = 48


In [86]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE IF JOB ARRAY IS RUNNING
# recombine=True

In [87]:
def Recombine(PROCESSING,num_jobs):
    dir2=dir+'Project_Algorithms/Entrainment/SBATCH/job_out_3/'
    dir3=dir+'Project_Algorithms/Entrainment/OUTPUT/'
    
    if PROCESSING==False:
        out_file=dir3+f'2D_entrainmentdetrainment_combined_profiles_{res}_{t_res}_{Np_str}.h5'
    elif PROCESSING==True:
        out_file=dir3+f'2D_entrainmentdetrainment_combined_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}.h5'
    
    vars=["profile_array_c_to_g_E","profile_array_g_to_c_E","profile_array_c_to_g_D","profile_array_g_to_c_D"]
    InitiateArray(out_file,vars,t_chunk_size=50,z_chunk_size=2)
    
    with h5py.File(out_file, 'r+') as f_out:
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
            [a,b,_]=StartJobArray(job_id,num_jobs)
    
            if PROCESSING==False:
                in_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
            elif PROCESSING==True:
                in_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}_{job_id}.h5'
            with h5py.File(in_file, 'r') as f_in: 
                for var in vars:
                    f_out[var][a:b]=f_in[var][:]

In [88]:
if recombine==True:
    Recombine(PROCESSING,num_jobs=num_jobs)

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [ ]:
################################################
#TESTING TESTING 

In [54]:
# #TESTING COMBINED ENTRAINMENT RATIO
# [start_slurm_job,end_slurm_job]=StartSlurmJobArray(num_jobs=num_jobs,num_slurm_jobs=num_slurm_jobs,ISRUN=False) #if ISRUN is False, then will not run using slurm_job_array

# print(f"Running on Slurm_Jobs for Slurm_Job_Ids: {(start_slurm_job,end_slurm_job-1)}")

# job_id_list=np.arange(start_slurm_job,end_slurm_job)
# for job_id in job_id_list[500:]:
#     if job_id % 1 == 0: print(f'current job_id = {job_id}')
#     [start_job,end_job,index_adjust]=StartJobArray(job_id,num_jobs)

#     #SLICING DATA
#     [data,parcel]=GetData(data1,parcel1,start_job,end_job)
#     [A_g,A_c]=GetAData(PROCESSING,start_job,end_job)
#     [A_g_Prior,A_c_Prior]=GetAPrior(PROCESSING,start_job) #FIX FOR FIRST TIMESTEP
#     [A_g_Post,A_c_Post]=GetAPost(PROCESSING,end_job) #FIX FOR LAST TIMESTEP #*#*#
#     [T,Z]=GetSpatialData(data,start_job,end_job)

#     #RUNNING CALCULATION
#     VARs=RunCalculation(data, A_g,A_c, T,Z, A_g_Prior,A_c_Prior,A_g_Post,A_c_Post)

#     #SAVING
#     SaveProfile(VARs,job_id,PROCESSING)
#     # check_memory(globals())
#     # del VARs
#     break


Running on Slurm_Jobs for Slurm_Job_Ids: (1, 600)
current job_id = 501


In [29]:
# plotting=True
# #data=data1 #parcel=parcel1


# if plotting==True:
#     #Calculate Mass Constant
#     # calculate='single_time'
#     # calculate=True
#     calculate=False
    
#     if calculate==True:
#         Nt=len(data['time'])
#         m_arr=np.zeros((Nt))
#         for t in np.arange(Nt):
#             if np.mod(t,25)==0: print(t)
#             m_arr[t]=m(t)
#         dir3=dir+f'Project_Algorithms/Entrainment/OUTPUT/'
#         np.save(dir3+f'Mass_Array_{res}_{t_res}_{Np_str}.npy', m_arr)
#     elif calculate=='single_time':
#         Nt=len(data['time'])
#         m_arr=np.zeros((Nt))
    
#         t=0 #len(data['time'])//2 #Pick some middle time
#         m_300=m(t)
#         for t in np.arange(Nt):
#             m_arr[t]=m_300 #UNCOMMENT FOR FULL CALCULATION
#         dir3=dir+f'Project_Algorithms/Entrainment/OUTPUT/'
#         np.save(dir3+f'Mass_Array_{res}_{t_res}_{Np_str}.npy', m_arr)
#     else:
#         dir3=dir+f'Project_Algorithms/Entrainment/OUTPUT/'
#         m_arr = np.load(dir3+f'Mass_Array_{res}_{t_res}_{Np_str}.npy')
    
#     # # TESTING
#     # lst=[]
#     # for t in np.arange(133):
#     #     lst.append(m_arr[t])
    
#     # plt.plot(lst)
#     # (np.max(lst)-np.min(lst))*100/np.mean(lst)

# if plotting==True:
#     #constants
#     Cp=1004 #Jkg-1K-1
#     Cv=717 #Jkg-1K-1
#     Rd=Cp-Cv #Jkg-1K-1
#     eps=0.608
    
#     Lx=(data['xf'][-1].item()-data['xf'][0].item())*1000 #x length (m)
#     Ly=(data['yf'][-1].item()-data['yf'][0].item())*1000 #y length (m)
#     Np=len(parcel['xh']) #number of lagrangian parcles
#     dt=(data['time'][1]-data['time'][0]).item()/1e9 #sec
#     dx=(data['xf'][1].item()-data['xf'][0].item())*1e3 #meters
#     dy=(data['yf'][1].item()-data['yf'][0].item())*1e3 #meters
#     xs=data['xf'].values*1000
#     ys=data['yf'].values*1000
#     zs=data['zf'].values*1000
    
#     def zf(z):
#         k=z #z is the # level of z
#         out=data['zf'].values[k]*1000
        
#         return out
#     # def rho(x,y,z,t):
#     #     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
#     #     p0=101325 #Pa
#     #     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
#     #     T=theta*(p/p0)**(Rd/Cp)
#     #     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
#     #     # Tv=T*(1+eps*qv)
#     #     Tv=T*(eps+qv)/(eps*(1+qv))
#     #     rho = p/(Rd*Tv)
#     #     out=rho
#     #     return out
    
#     def rho(x,y,z,rho_data_t):
#         out=rho_data_t[z,y,x]
#         return out
#     def m(t):
#         rho_data_t=data['rho'].isel(time=t).data
        
#         m=0
#         #triple sum
#         for k in range(len(data['zh'])):
#             dz=(zf(k+1)-zf(k))
#             for j in range(len(data['yh'])):
#                 for i in range(len(data['xh'])):
#                     rho_out=rho(i,j,k,rho_data_t)
#                     m+=rho_out*dz
                    
#         #triple sum
#         out=m*dx*dy/Np
#         return out


# plotting=True
# if plotting==True:
#     PROCESSING=False
#     PROCESSING=True
    
#     dir3=dir+'Project_Algorithms/Entrainment/SBATCH/job_out_3/'
#     if PROCESSING==False:
#         open_file=dir3+f'2D_entrainmentdetrainment_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
#     if PROCESSING==True:
#         open_file=dir3+f'2D_entrainmentdetrainment_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}_{job_id}.h5'
#     with h5py.File(open_file, "r") as h5f:
#         profile_array_e_g = h5f["profile_array_e_g"][:]
#         profile_array_e_c = h5f["profile_array_e_c"][:]
#         profile_array_d_g = h5f["profile_array_d_g"][:]
#         profile_array_d_c = h5f["profile_array_d_c"][:]


# if plotting==True:
#     def apply_constant(profile_array,apply):
#         if apply==True:
#             Nt=profile_array.shape[0]
#             Nz=profile_array.shape[1]
        
#             profile_array/=(Lx*Ly*dt)
#             for t in np.arange(Nt):
#                 profile_array[t]*=m_arr[t]
#             for z in np.arange(Nz):
#                 dz=zf(z+1)-zf(z)
#                 profile_array[:,z]/=dz
#         return profile_array
#     #APPLY CONSTANTS TO ENTRAINMENT VALUE
#     ##################################################
#     profile_array_e_g=apply_constant(profile_array_e_g,apply=True)
#     profile_array_e_c=apply_constant(profile_array_e_c,apply=True)
#     profile_array_d_g=-apply_constant(profile_array_d_g,apply=True)
#     profile_array_d_c=-apply_constant(profile_array_d_c,apply=True)
#     ##################################################

In [55]:
# e_g=np.mean(profile_array_e_g)
# e_c=np.mean(profile_array_e_c)
# d_g=np.mean(profile_array_d_g)
# d_c=np.mean(profile_array_d_c)

# profile_array_c_to_g_E=VARs['profile_array_c_to_g_E']
# profile_array_c_to_g_D=VARs['profile_array_c_to_g_D']
# c_to_g_E=apply_constant(profile_array_c_to_g_E,apply=True)
# c_to_g_D=apply_constant(profile_array_c_to_g_D,apply=True)
# c_to_g_E=np.mean(c_to_g_E,axis=0)
# c_to_g_D=np.mean(c_to_g_D,axis=0)

# profile_array_g_to_c_E=VARs['profile_array_g_to_c_E']
# profile_array_g_to_c_D=VARs['profile_array_g_to_c_D']
# g_to_c_E=apply_constant(profile_array_g_to_c_E,apply=True)
# g_to_c_D=apply_constant(profile_array_g_to_c_D,apply=True)
# g_to_c_E=np.mean(g_to_c_E,axis=0)
# g_to_c_D=np.mean(g_to_c_D,axis=0)

In [88]:
# import matplotlib.pyplot as plt
# import numpy as np

# fig, axs = plt.subplots(1, 2, figsize=(8, 4), sharey=True)

# # ==== First subplot ====
# axs[0].plot(c_to_g_E / e_g, data['zh'], label='1')
# axs[0].plot(c_to_g_E / d_c, data['zh'], label='2')
# axs[0].plot(g_to_c_E / e_c, data['zh'], label='3')
# axs[0].plot(g_to_c_E / d_g, data['zh'], label='4')
# axs[0].axvline(1, color='k', linestyle='--')
# axs[0].legend()
# axs[0].set_title("Entrainment Ratios")

# # ==== Second subplot ====
# axs[1].plot(c_to_g_E / e_g, data['zh'], label='1')
# axs[1].plot(c_to_g_D / d_c, data['zh'], label='2')
# axs[1].plot(g_to_c_E / e_c, data['zh'], label='3')
# axs[1].plot(g_to_c_D / d_g, data['zh'], label='4')
# axs[1].axvline(1, color='k', linestyle='--')
# axs[1].legend()
# axs[1].set_title("Entrainment + Detrainment Ratios")

# # ==== Shared settings ====
# for ax in axs:
#     ax.set_xlabel("Ratio")
#     ax.grid(True)

# axs[0].set_ylabel("Height (m)")
# plt.tight_layout()
# plt.show()


In [ ]:
####################################################

In [1]:

# # dx = 1 km; Np = 1M; Nt = 5 min
# data1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
# parcel1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
# res='1km';t_res='5min'
# Np_str='1e6'

# job_id=44;num_jobs=60
# print(StartJobArray(job_id,num_jobs))

# # dx = 1km; Np = 50M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# job_id=148;num_jobs=200
# print(StartJobArray(job_id,num_jobs))

In [2]:
# #TESTING
# res='1km';t_res='5min';Np_str='1e6'

# job_id=44;num_jobs=60
# # print(StartJobArray(job_id,num_jobs))

# dir2=dir+'Project_Algorithms/Entrainment/job_out_3/'
# in_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}_{job_id}.h5'
# # in_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
# with h5py.File(in_file,'r') as f:
#     g_to_c=f['profile_array_g_to_c'][:]
# print(g_to_c)

# #####
# x=np.arange(g_to_c.shape[0])
# y=data1['zh'].data
# plt.contourf(x,y,g_to_c.T)
# plt.colorbar()

In [3]:
# #TESTING
# res='1km';t_res='1min';Np_str='50e6'

# job_id=148;num_jobs=200
# # print(StartJobArray(job_id,num_jobs))

# dir2=dir+'Project_Algorithms/Entrainment/job_out_3/'
# in_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}_{job_id}.h5'
# in_file=dir2+f'2D_entrainmentdetrainment_combined_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
# with h5py.File(in_file,'r') as f:
#     g_to_c=f['profile_array_g_to_c'][:]
# print(g_to_c)

# #####
# x=np.arange(g_to_c.shape[0])
# y=data1['zh'].data
# plt.contourf(x,y,g_to_c.T)
# plt.colorbar()